In [22]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [23]:
data = pd.read_csv("Final_data.csv")

In [24]:
data_x = data[["Gaze Velocity","Gaze Acceleration","Fixation Duration"]]
data_y = data[["Collision Time","Response_Time"]]

In [25]:
data_x

,Gaze Velocity,Gaze Acceleration,Fixation Duration
0,5.973269,-537.073195,800.0520
1,6.809780,109.675649,233.3456
2,29.070157,-389.920469,433.3288
3,14.827312,198.128872,200.0408
4,3.798489,-226.725038,1216.2088
...,...,...,...
79,4.006833,-90.672156,633.3824
80,4.006833,-90.672156,633.3824
81,4.006833,-90.672156,633.3824
82,4.006833,-90.672156,633.3824


In [26]:
# Selecting features and target variables
X = data[["Gaze Velocity", "Gaze Acceleration", "Fixation Duration"]].values
# y = data[["Collision Time", "Response_Time"]].values
y = data[["Response_Time"]].values

# Splitting the data into train and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [27]:
# Standardizing the features for better training performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [28]:
# Define the neural network model
class HazardNN(nn.Module):
    def __init__(self):
        super(HazardNN, self).__init__()
        self.layer1 = nn.Linear(3, 32)
        self.layer2 = nn.Linear(32, 128)
        self.output = nn.Linear(128, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.output(x)
        return x

In [29]:
# Initialize the model, loss function, and optimizer
model = HazardNN()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training the model
epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}')

# Evaluate the model
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensor)
    # print(y_pred)
    # print(y_test)
    test_loss = mean_squared_error(y_test, y_pred.numpy())
    test_loss_abs = mean_absolute_error(y_test, y_pred.numpy())

print(f'Test MSE Loss: {test_loss:.4f}')
print(f'Test MAE: {test_loss_abs:.4f}')

Epoch [10/100], Loss: 1.4340
Epoch [20/100], Loss: 1.1112
Epoch [30/100], Loss: 1.0490
Epoch [40/100], Loss: 0.9987
Epoch [50/100], Loss: 0.9769
Epoch [60/100], Loss: 0.9598
Epoch [70/100], Loss: 0.9442
Epoch [80/100], Loss: 0.9281
Epoch [90/100], Loss: 0.9122
Epoch [100/100], Loss: 0.8954
tensor([[0.8299],
        [0.7174],
        [1.2809],
        [1.2841],
        [0.9930],
        [0.8299],
        [0.4158],
        [1.0335],
        [0.5243],
        [0.8299],
        [1.2107],
        [1.2637],
        [1.0028],
        [0.8299],
        [0.4780],
        [0.8911],
        [1.0472]])
[[0.409]
 [2.505]
 [0.361]
 [1.254]
 [1.402]
 [2.756]
 [0.655]
 [1.003]
 [3.201]
 [0.404]
 [1.15 ]
 [1.255]
 [0.804]
 [0.354]
 [1.96 ]
 [0.993]
 [0.097]]
Test MSE Loss: 1.1106
Test MAE: 0.7144


In [30]:
# from sklearn.linear_model import ridge_regression
# from sklearn.metrics import mean_squared_error
#
# # Train an MLP model using scikit-learn
# mlp_model = ridge_regression(hidden_layer_sizes=(16, 8), activation='relu', solver='adam', max_iter=500)
# mlp_model.fit(X_train, y_train)
#
# # Predict and evaluate
# y_pred = mlp_model.predict(X_test)
# test_loss = mean_squared_error(y_test, y_pred)
#
# print(f'Scikit-learn MLP Test MSE Loss: {test_loss:.4f}')


In [31]:
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error

# Selecting features and target variable (assuming prediction of 'Collision Time')
X = data[["Gaze Velocity", "Gaze Acceleration", "Fixation Duration"]].values
y = data["Collision Time"].values  # Predicting only 'Collision Time'

# Splitting the data into train and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardizing the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create and train the Gaussian SVM model
svm_model = SVR(kernel='rbf',  epsilon=0.1, gamma='scale')  # Gaussian kernel (rbf)
svm_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = svm_model.predict(X_test_scaled)

# Evaluate the model using Mean Squared Error
test_mse = mean_squared_error(y_test, y_pred)

# Display the results
print(f'Test MSE Loss: {test_mse:.4f}')


Test MSE Loss: 1.3235


In [32]:
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
y_scaled = scaler.fit_transform(y.reshape(-1, 1)).ravel()

# Define parameter grid
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1, 1],
    'epsilon': [0.01, 0.1, 0.5, 1]
}

# Perform Grid Search
svr = SVR(kernel='rbf')
grid_search = GridSearchCV(svr, param_grid, scoring='neg_mean_squared_error', cv=5)
grid_search.fit(X_scaled, y_scaled)

# Get best parameters
best_svr = grid_search.best_estimator_
print("Best parameters:", grid_search.best_params_)

# Predict using the optimized model
y_pred_scaled = best_svr.predict(X_scaled)
y_pred = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

# Evaluate performance
mse = mean_squared_error(y, y_pred)
print(f"Optimized Test MSE: {mse:.4f}")


Best parameters: {'C': 100, 'epsilon': 0.5, 'gamma': 0.01}
Optimized Test MSE: 8.4190
